# 07 - 检索器与路由（Retrievers & Routing）

## 学习目标
- 掌握 SelfQueryRetriever 自动提取查询和元数据过滤
- 使用 ContextualCompressionRetriever 压缩和过滤检索结果
- 使用 MultiQueryRetriever 生成多角度查询提升召回率
- 组合稠密+稀疏检索器（EnsembleRetriever）
- 自定义检索器：子类化 BaseRetriever
- 使用 RunnableBranch 实现智能路由

In [ ]:
# 安装必要依赖（如未安装请取消注释）
# !pip install langchain langchain-community langchain-openai langchain-chroma chromadb lark

## 1. SelfQueryRetriever - 自查询检索器

SelfQueryRetriever 使用 LLM 从自然语言查询中提取：
1. **语义查询** (query string) - 用于向量相似度搜索
2. **元数据过滤器** (metadata filter) - 基于文档属性过滤

In [ ]:
from langchain_core.documents import Document

# 准备测试数据 - 带有丰富元数据的电影数据集
movies = [
    Document(
        page_content="一部关于人工智能觉醒的科幻电影，探讨了意识和伦理问题",
        metadata={"title": "AI觉醒", "year": 2023, "genre": "科幻", "rating": 8.5, "director": "张导演"}
    ),
    Document(
        page_content="一个跨越三十年的爱情故事，发生在上海和纽约两个城市",
        metadata={"title": "双城之恋", "year": 2022, "genre": "爱情", "rating": 7.8, "director": "李导演"}
    ),
    Document(
        page_content="讲述年轻程序员创业的喜剧片，充满了科技行业的笑料",
        metadata={"title": "代码人生", "year": 2024, "genre": "喜剧", "rating": 7.2, "director": "王导演"}
    ),
    Document(
        page_content="悬疑推理电影，一位侦探调查离奇的连环失踪案",
        metadata={"title": "失踪者", "year": 2023, "genre": "悬疑", "rating": 8.1, "director": "赵导演"}
    ),
    Document(
        page_content="基于真实历史事件改编的战争史诗巨制",
        metadata={"title": "战火岁月", "year": 2024, "genre": "战争", "rating": 9.0, "director": "张导演"}
    ),
    Document(
        page_content="一部探讨人工智能伦理的纪录片",
        metadata={"title": "AI伦理", "year": 2023, "genre": "纪录片", "rating": 8.8, "director": "李导演"}
    ),
    Document(
        page_content="轻松愉快的动画电影，适合全家观看的冒险故事",
        metadata={"title": "森林冒险", "year": 2022, "genre": "动画", "rating": 7.5, "director": "王导演"}
    ),
    Document(
        page_content="恐怖片，讲述一个古老村庄的诅咒传说",
        metadata={"title": "村庄诅咒", "year": 2023, "genre": "恐怖", "rating": 6.8, "director": "赵导演"}
    ),
]

print(f"准备了 {len(movies)} 部电影数据")
genres = set(m.metadata["genre"] for m in movies)
years = set(m.metadata["year"] for m in movies)
print(f"类型: {genres}")
print(f"年份: {sorted(years)}")
print(f"评分范围: {min(m.metadata['rating'] for m in movies)} - {max(m.metadata['rating'] for m in movies)}")

In [ ]:
print("=== SelfQueryRetriever 概念与代码 ===\n")

# SelfQueryRetriever 的核心组件:
# 1. AttributeInfo - 描述每个元数据字段的信息
# 2. LLM - 用于解析自然语言查询
# 3. 向量存储 - 底层检索引擎

from langchain.chains.query_constructor.base import AttributeInfo

# 定义元数据字段描述
metadata_field_info = [
    AttributeInfo(
        name="genre",
        description="电影的类型",
        type="string",
    ),
    AttributeInfo(
        name="year",
        description="电影上映的年份",
        type="integer",
    ),
    AttributeInfo(
        name="rating",
        description="电影的评分（1-10分）",
        type="float",
    ),
    AttributeInfo(
        name="director",
        description="电影的导演姓名",
        type="string",
    ),
]

# 文档内容描述
document_content_description = "电影的中文简介"

print("元数据字段定义:")
for info in metadata_field_info:
    print(f"  - {info.name} ({info.type}): {info.description}")

print("\n" + "=" * 60)

print("# SelfQueryRetriever 工作流程:")
print("# 输入: '2023年的高分科幻电影'")
print("#   -> LLM 解析出:")
print("#      query = '科幻电影'")
print("#      filter = AND(year == 2023, genre == '科幻', rating >= 8)")
print("#   -> 向量存储执行:")
print("#      vectorstore.similarity_search('科幻电影', filter={...})")
print("# 输出: 符合条件的文档列表")

print("\n" + "=" * 60)

print("# 完整代码示例:")
print("""
from langchain.retrievers import SelfQueryRetriever
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

# 创建向量存储
embeddings = OpenAIEmbeddings()
vectorstore = Chroma.from_documents(
    documents=movies,
    embedding=embeddings,
)

# 创建 LLM
llm = ChatOpenAI(model="gpt-4", temperature=0)

# 创建 SelfQueryRetriever
retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    document_contents=document_content_description,
    metadata_field_info=metadata_field_info,
    enable_limit=True,                # 允许 LIMIT 比较
    verbose=True,                     # 打印解析过程
)

# 自然语言查询 - 自动提取过滤条件
results = retriever.invoke("2023年评分高于8分的科幻或悬疑电影")
for doc in results:
    print(f"{doc.metadata['title']} ({doc.metadata['year']}) - {doc.metadata['rating']}分")
""")

print("\n# 比较运算符支持（通过 enable_limit 和字段类型控制）:")
print("# - 字符串: eq (等于), ne (不等于)")
print("# - 数值: eq, ne, gt, gte, lt, lte")
print("# - 日期: eq, ne, gt, gte, lt, lte")
print("# - 列表: in, nin (不在列表中)")
print("# - 逻辑: AND, OR 组合")
print()
print("# 启用/禁用特定运算符:")
print("# retriever = SelfQueryRetriever.from_llm(")
print("#     ...,")
print("#     enable_limit=False,          # 禁用 LIMIT")
print("#     search_kwargs={'k': 5},      # 固定返回数量")
print("# )")

## 2. ContextualCompressionRetriever - 上下文压缩检索器

在检索完成后，对文档进行压缩或过滤，保留最相关的部分。

In [ ]:
print("=== ContextualCompressionRetriever ===\n")

print("# 两种压缩策略:")
print()

print("策略1: LLMChainExtractor - 从每个文档中提取仅与查询相关的部分")
print("-" * 60)
print("from langchain.retrievers import ContextualCompressionRetriever")
print("from langchain.retrievers.document_compressors import LLMChainExtractor")
print()
print("# 创建压缩器")
print("compressor = LLMChainExtractor.from_llm(llm)")
print()
print("# 包装检索器")
print("compression_retriever = ContextualCompressionRetriever(")
print("    base_compressor=compressor,")
print("    base_retriever=base_retriever,")
print(")")
print()
print("# 效果: 原文档可能500字，压缩后只保留与查询相关的100字")

print("\n" + "=" * 60)

print("策略2: LLMChainFilter - 过滤掉与查询完全无关的文档")
print("-" * 60)
print("from langchain.retrievers.document_compressors import LLMChainFilter")
print()
print("# 创建过滤器")
print("_filter = LLMChainFilter.from_llm(llm)")
print()
print("# 包装检索器")
print("filter_retriever = ContextualCompressionRetriever(")
print("    base_compressor=_filter,")
print("    base_retriever=base_retriever,")
print(")")
print()
print("# 效果: 检索10个文档，过滤后只保留5个真正相关的")

print("\n" + "=" * 60)

print("# 完整的 before/after 对比示例:")
print("""
# === Before: 普通检索器 ===
base_retriever = vectorstore.as_retriever(search_kwargs={'k': 5})
query = "人工智能在电影中的表现"

before_results = base_retriever.invoke(query)
print("=== 压缩前 ===")
for i, doc in enumerate(before_results):
    print(f"[{i+1}] ({len(doc.page_content)}字) {doc.page_content[:100]}...")

# === After: 使用 LLMChainExtractor 压缩 ===
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

compressor = LLMChainExtractor.from_llm(llm)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever,
)

after_results = compression_retriever.invoke(query)
print("\\n=== 压缩后 ===")
for i, doc in enumerate(after_results):
    print(f"[{i+1}] ({len(doc.page_content)}字) {doc.page_content[:100]}...")
""")

print("\n# 其他内置压缩器:")
print("# - EmbeddingsFilter: 基于嵌入相似度过滤（更便宜）")
print("# - DocumentCompressorPipeline: 组合多个压缩器")
print("")
print("from langchain.retrievers.document_compressors import EmbeddingsFilter")
print("embeddings_filter = EmbeddingsFilter(")
print("    embeddings=embeddings,")
print("    similarity_threshold=0.76,")
print(")")

## 3. MultiQueryRetriever - 多查询检索器

使用 LLM 从不同角度生成多个查询变体，对每个变体分别检索，最后合并去重。
这解决了查询措辞不佳导致的检索效果差的问题。

In [ ]:
print("=== MultiQueryRetriever ===\n")

print("# MultiQueryRetriever 的工作原理:")
print("# 1. 输入原始查询")
print("# 2. LLM 生成多个重新表述的查询")
print("# 3. 对每个查询分别执行检索")
print("# 4. 合并所有结果并去重")
print()

print("示例流程:")
print("  原始查询: '怎样让AI记住更多信息？'")
print("  LLM生成:")
print("    1. 'AI系统的记忆增强方法'")
print("    2. '如何扩展语言模型的上下文长度'")
print("    3. '向量数据库在AI记忆中的应用'")
print("    4. '检索增强生成中的知识存储策略'")
print()
print("  每个变体单独检索 -> 合并去重 -> 返回")

print("\n" + "=" * 60)

print("# 完整代码:")
print("""
from langchain.retrievers import MultiQueryRetriever
from langchain_openai import ChatOpenAI

# 创建基础检索器
base_retriever = vectorstore.as_retriever(search_kwargs={'k': 3})

# 创建 LLM 用于生成查询变体
llm = ChatOpenAI(model="gpt-4", temperature=0.7)  # 高温度增加多样性

# 创建多查询检索器
multi_retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=llm,
    include_original=True,        # 是否包含原始查询的结果
)

# 查看生成的多个查询
import logging
logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

unique_docs = multi_retriever.invoke("AI技术的发展趋势")
print(f"多查询检索返回 {len(unique_docs)} 个去重后的文档")
""")

print("\n# 对比单查询 vs 多查询:")
print()
print("# 单查询检索:")
print("docs_single = base_retriever.invoke('AI技术的发展趋势')")
print("print(f'单查询: {len(docs_single)} 个结果')")
print()
print("# 多查询检索:")
print("docs_multi = multi_retriever.invoke('AI技术的发展趋势')")
print("print(f'多查询: {len(docs_multi)} 个去重结果')")
print()
print("# 多查询通常能召回更多相关文档，但需要更多 LLM 调用")
print("# 适用于: 查询难以准确表述的领域; 需要高召回率的场景")

print("\n# 自定义提示词:")
print("from langchain.retrievers.multi_query import MultiQueryRetriever")
print("from langchain_core.prompts import PromptTemplate")
print()
print("custom_prompt = PromptTemplate(")
print("    input_variables=['question'],")
print("    template='''你是一个查询改写专家。请将以下问题改写为3个不同视角的查询：")
print("        问题: {question}")
print("        改写查询:")
print("    '''")
print(")")

## 4. EnsembleRetriever - 集成检索器

组合稠密检索（向量相似度）和稀疏检索（BM25关键词），加权合并结果。
结合了两者的优势：语义理解 + 关键词精确匹配。

In [ ]:
print("=== EnsembleRetriever ===\n")

print("# EnsembleRetriever 设计思想:")
print("# 稠密检索 (Dense): 语义相似度，能找到意思相近但不含关键词的文档")
print("# 稀疏检索 (Sparse/BM25): 关键词匹配，擅长精确匹配和罕见词")
print("# 集成检索: 加权合并，取长补短")
print()

print("# 完整代码:")
print("""
from langchain.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever

# 1. 稠密检索器 (Dense - 向量相似度)
dense_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5},
)

# 2. 稀疏检索器 (Sparse - BM25 关键词)
bm25_retriever = BM25Retriever.from_documents(
    documents=all_documents,
)
bm25_retriever.k = 5

# 3. 集成检索器 - 加权组合
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, dense_retriever],
    weights=[0.3, 0.7],  # BM25权重0.3, 稠密权重0.7
    c=60,                 # 混合参数
    id_key="doc_id",      # 用于去重的标识键
)

# 检索
results = ensemble_retriever.invoke("LangChain的核心概念")
for doc in results:
    print(f"{doc.page_content[:80]}...")
""")

print("\n# 权重调优指南:")
print("# - [0.5, 0.5]: 平衡语义和关键词")
print("# - [0.8, 0.2]: 关键词优先（适用于术语密集领域）")
print("# - [0.2, 0.8]: 语义优先（适用于对话式查询）")
print()

print("# 结合更多检索器:")
print("# ensemble_retriever = EnsembleRetriever(")
print("#     retrievers=[")
print("#         bm25_retriever,     # BM25")
print("#         dense_retriever,    # 稠密向量")
print("#         multi_retriever,    # 多查询")
print("#     ],")
print("#     weights=[0.2, 0.5, 0.3],")
print("# )")

print("\n# 实际对比效果:")
print("print('=== 仅稠密检索 ===')")
print("dense_results = dense_retriever.invoke(query)")
print()
print("print('=== 仅BM25检索 ===')")
print("sparse_results = bm25_retriever.invoke(query)")
print()
print("print('=== 集成检索（稠密+稀疏）===')")
print("ensemble_results = ensemble_retriever.invoke(query)")
print("# 集成结果通常包含稠密和稀疏各自的优势")

## 5. 自定义检索器 - 子类化 BaseRetriever

当内置检索器不满足需求时，可以通过继承 BaseRetriever 实现自定义检索逻辑。

In [ ]:
from langchain_core.retrievers import BaseRetriever
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from langchain_core.documents import Document
from typing import List

class KeywordMemoryRetriever(BaseRetriever):
    """
    自定义检索器: 结合关键词匹配和最近历史记忆。
    
    工作流程:
    1. 从查询中提取关键词
    2. 在文档库中执行关键词匹配
    3. 结合用户最近的查询历史提升相关文档权重
    4. 返回排序后的文档列表
    """
    
    documents: List[Document] = []
    history: List[str] = []
    k: int = 4
    history_boost: float = 1.2  # 历史相关文档的权重提升系数
    
    class Config:
        arbitrary_types_allowed = True
    
    def _get_relevant_documents(
        self, 
        query: str, 
        *, 
        run_manager: CallbackManagerForRetrieverRun = None
    ) -> List[Document]:
        """
        核心检索方法 - 必须实现。
        
        Args:
            query: 用户查询字符串
            run_manager: 回调管理器（用于追踪和日志）
            
        Returns:
            相关文档列表
        """
        # 步骤1: 提取关键词
        keywords = self._extract_keywords(query)
        
        # 步骤2: 关键词匹配评分
        scored_docs = []
        for doc in self.documents:
            score = self._compute_keyword_score(doc.page_content, keywords)
            if score > 0:
                scored_docs.append((doc, score))
        
        # 步骤3: 历史提升 - 如果查询与历史相关
        if self._is_related_to_history(query):
            scored_docs = [
                (doc, score * self.history_boost) 
                for doc, score in scored_docs
            ]
        
        # 步骤4: 排序并返回 top-k
        scored_docs.sort(key=lambda x: x[1], reverse=True)
        top_docs = [doc for doc, _ in scored_docs[:self.k]]
        
        # 步骤5: 记录历史
        self.history.append(query)
        if len(self.history) > 10:
            self.history.pop(0)
        
        return top_docs
    
    def _extract_keywords(self, query: str) -> List[str]:
        """简单关键词提取 - 实际应用中可使用 NLP 库"""
        # 移除常见停用词
        stopwords = {'的', '了', '是', '在', '和', '与', '或', '一个', '这个', '那个', '吗', '呢'}
        words = query.replace('？', ' ').replace('？', ' ').split()
        return [w for w in words if w not in stopwords and len(w) > 1]
    
    def _compute_keyword_score(self, content: str, keywords: List[str]) -> float:
        """计算关键词匹配分数"""
        if not keywords:
            return 0.0
        matches = sum(1 for kw in keywords if kw in content)
        return matches / len(keywords)
    
    def _is_related_to_history(self, query: str) -> bool:
        """检查查询是否与近期历史相关"""
        if not self.history:
            return False
        query_words = set(query)
        for past_query in self.history[-3:]:
            past_words = set(past_query)
            overlap = len(query_words & past_words)
            if overlap > 0:
                return True
        return False


# 测试自定义检索器
print("=== 自定义检索器测试 ===\n")

test_documents = [
    Document(page_content="LangChain 是一个用于开发 LLM 应用的框架", metadata={"id": "doc1"}),
    Document(page_content="Python 是最受欢迎的编程语言之一", metadata={"id": "doc2"}),
    Document(page_content="LangChain 支持多种向量数据库集成", metadata={"id": "doc3"}),
    Document(page_content="RAG 系统使用检索增强生成技术", metadata={"id": "doc4"}),
    Document(page_content="Python 在数据科学和人工智能领域广泛应用", metadata={"id": "doc5"}),
    Document(page_content="LangChain 的 LCEL 语法让链式调用更简洁", metadata={"id": "doc6"}),
]

custom_retriever = KeywordMemoryRetriever(
    documents=test_documents,
    k=3,
    history_boost=1.5,
)

# 第一次查询
query1 = "LangChain框架介绍"
results1 = custom_retriever.invoke(query1)
print(f"查询: '{query1}'")
print(f"返回 {len(results1)} 个文档:")
for i, doc in enumerate(results1):
    print(f"  [{i+1}] {doc.page_content[:60]}...")

# 第二次查询（相关查询，历史会提升分数）
query2 = "LangChain向量数据库"
results2 = custom_retriever.invoke(query2)
print(f"\n查询: '{query2}' (与历史相关)")
print(f"返回 {len(results2)} 个文档:")
for i, doc in enumerate(results2):
    print(f"  [{i+1}] {doc.page_content[:60]}...")

# 不相关查询
query3 = "数据科学人工智能"
results3 = custom_retriever.invoke(query3)
print(f"\n查询: '{query3}'")
print(f"返回 {len(results3)} 个文档:")
for i, doc in enumerate(results3):
    print(f"  [{i+1}] {doc.page_content[:60]}...")

print(f"\n检索历史: {custom_retriever.history}")

print("\n# 自定义检索器的关键点:")
print("# 1. 必须继承 BaseRetriever")
print("# 2. 必须实现 _get_relevant_documents(query, *, run_manager)")
print("# 3. 可以使用 Pydantic 字段定义配置参数")
print("# 4. 支持回调管理器进行追踪")
print("# 5. 可以与其他 LangChain 组件无缝组合")

## 6. 路由 - RunnableBranch

RunnableBranch 根据条件将查询路由到不同的检索器或处理链。
例如：法律问题 -> 法律检索器；技术问题 -> 技术文档检索器。

In [ ]:
from langchain_core.runnables import RunnableBranch, RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

print("=== RunnableBranch 路由 ===\n")

# 定义不同类型的检索器
def create_fake_retriever_returning(docs: List[str], name: str):
    """创建一个返回预设文档的模拟检索器"""
    async def _retrieve(query: str) -> List[Document]:
        # 这里应该是真实的检索逻辑
        return [Document(page_content=d, metadata={"source": name}) for d in docs]
    return _retrieve

# 创建模拟检索器
code_retriever = RunnableLambda(lambda q: [
    Document(page_content=f"代码库搜索结果: Python代码示例 def hello(): print('world')",
             metadata={"source": "code"}),
])

docs_retriever = RunnableLambda(lambda q: [
    Document(page_content=f"文档搜索结果: LangChain是一个LLM应用框架",
             metadata={"source": "docs"}),
])

general_retriever = RunnableLambda(lambda q: [
    Document(page_content=f"通用搜索结果: {q} 的相关信息",
             metadata={"source": "general"}),
])

# 路由条件函数: 判断查询类型
def detect_query_type(query: str) -> str:
    """检测查询类型"""
    query_lower = query.lower()
    code_keywords = ["代码", "code", "函数", "function", "编程", "programming", 
                     "python", "javascript", "实现", "example"]
    doc_keywords = ["文档", "document", "说明", "教程", "tutorial", "api",
                    "指南", "guide", "介绍", "introduction"]
    
    code_score = sum(1 for kw in code_keywords if kw in query_lower)
    doc_score = sum(1 for kw in doc_keywords if kw in query_lower)
    
    if code_score > doc_score:
        return "code"
    elif doc_score > 0:
        return "docs"
    else:
        return "general"

# 使用 RunnableBranch 创建路由
branch = RunnableBranch(
    (lambda x: detect_query_type(x) == "code", code_retriever),
    (lambda x: detect_query_type(x) == "docs", docs_retriever),
    general_retriever,  # 默认分支
)

# 测试路由
test_queries = [
    "Python代码示例",
    "LangChain使用文档",
    "今天天气怎么样",
    "如何实现RAG系统",
    "API调用指南",
]

print("路由测试:")
print("-" * 60)
for query in test_queries:
    query_type = detect_query_type(query)
    results = branch.invoke(query)
    if results:
        source = results[0].metadata.get("source", "unknown")
        print(f"查询: '{query}' -> 路由到: {query_type} -> 来源: {source}")

print("\n" + "=" * 60)

print("# RunnableBranch 在 LCEL 链中的使用:")
print("""
from langchain_core.runnables import RunnableBranch

# 定义处理链
code_chain = code_retriever | code_formatter | code_prompt | llm
docs_chain = docs_retriever | doc_formatter | doc_prompt | llm
general_chain = general_retriever | general_prompt | llm

# 创建路由链
full_chain = RunnableBranch(
    (lambda x: detect_query_type(x) == "code", code_chain),
    (lambda x: detect_query_type(x) == "docs", docs_chain),
    general_chain,
)

# 使用
result = full_chain.invoke("如何写一个Python排序函数？")
""")

print("\n# 路由策略对比:")
print("# - RunnableBranch: 基于条件的分支路由")
print("# - 语义路由: 让 LLM 根据查询内容选择")
print("# - 分类路由: 使用文本分类模型")
print("# - 评分路由: 多个检索器同时检索，选择得分最高的")

print("\n# 最佳实践:")
print("# 1. 为每个检索器配置不同的搜索参数")
print("# 2. 使用 RunnableBranch 的默认分支兜底")
print("# 3. 在路由条件中使用 try-except 防止异常")
print("# 4. 记录路由决策用于调试和分析")
print("# 5. 定期评估路由准确率并调整条件逻辑")

## 7. 检索策略对比总结

对本章涉及的所有检索策略进行对比。

In [ ]:
print("=== 检索策略全面对比 ===\n")

strategies = [
    ("基础相似度", "最简单的余弦相似度检索", "通用场景", "低", "低", "不支持元数据过滤"),
    ("SelfQuery", "LLM提取查询+过滤条件", "结构化元数据", "中", "中(LLM调用)", "依赖LLM解析准确性"),
    ("MultiQuery", "生成多个查询变体", "模糊查询", "中", "高", "多次LLM调用"),
    ("上下文压缩", "LLM提取/过滤文档", "长文档", "高", "高", "压缩可能丢失信息"),
    ("Ensemble(稠密+稀疏)", "加权组合多种检索", "混合场景", "低", "低", "需要调权"),
    ("MMR", "最大边际相关性", "需要多样性", "低", "低", "可能降低相关性"),
    ("自定义检索器", "完全自定义逻辑", "特殊需求", "不定", "不定", "需要开发维护"),
    ("路由(RunnableBranch)", "根据查询类型路由", "多领域", "低", "低", "需要定义路由规则"),
]

print(f"{'策略':<20} {'描述':<28} {'适用场景':<14} {'复杂度':<8} {'延迟':<8} {'注意事项':<25}")
print("-" * 105)
for name, desc, scenario, complexity, latency, notes in strategies:
    print(f"{name:<20} {desc:<28} {scenario:<14} {complexity:<8} {latency:<8} {notes:<25}")

print("\n# 选择流程:")
print("1. 确定是否需要元数据过滤 -> 需要: SelfQueryRetriever")
print("2. 确定是否需要查询多样性 -> 需要: MultiQueryRetriever")
print("3. 确定是否需要压缩长文档 -> 需要: ContextualCompressionRetriever")
print("4. 确定是否需要关键词+语义 -> 需要: EnsembleRetriever")
print("5. 确定是否需要多领域路由 -> 需要: RunnableBranch")
print("6. 以上都不满足 -> 自定义 BaseRetriever")